# 02 · Alineación y variables
**Proyecto Integrador ML2 — Segunda entrega · Predicción de la TRM**

**Objetivo (rúbricas 7.1 y 7.2).** Unir las cinco series en una sola tabla **sin fuga de información**, construir el objetivo y las 6 variables de la Entrega 1, probar que ninguna mira el futuro y fijar la partición cronológica entre desarrollo y test.

**Entrada:** `data/processed/series_limpias.csv` (notebook 01). **Salidas:** `data/processed/dataset.csv` y `data/processed/particion.json`.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
from IPython.display import display

from src import config as cfg
from src.alineacion import alinear, verificar_sin_fuga, tabla_relleno, REZAGO_PUBLICACION
from src.variables import agregar_variables, construir_dataset, prueba_perturbacion, REZAGOS_TRM
from src.validacion import particion_cronologica
from src.graficos import guardar_tabla

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
limpias = pd.read_csv(cfg.DATA_PROC / "series_limpias.csv", parse_dates=["fecha", "vigencia_hasta"])
serie = lambda n: limpias.loc[limpias["serie"] == n, ["fecha", "valor"]].reset_index(drop=True)
trm = serie("TRM")
macros = {"brent": serie("Brent"), "dxy": serie("DXY"), "tpm": serie("TPM"), "dff": serie("DFF")}
print(f"TRM publicadas: {len(trm):,} | periodo {cfg.FECHA_INICIO} a {cfg.FECHA_FIN}")

TRM publicadas: 5,412 | periodo 2003-12-01 a 2026-09-29


## 1. El problema: ¿qué se sabe y cuándo?
La fila *t* es una TRM publicada, con fecha igual al primer día de su vigencia. Se quiere predecir la **siguiente** TRM, que se calcula con las operaciones del mercado cambiario durante el día *t*. Por eso, en la fila *t* solo puede entrar información **disponible antes de que empiece el día *t***.

Cada serie se publica en un momento distinto, y de ahí sale una regla para cada una (**Tabla 2**):

In [2]:
disponibilidad = pd.DataFrame([
    ["TRM_t", "La tarde anterior a t (se calcula con las operaciones de t−1)", "Se usa la de la propia fila t"],
    ["Brent", "Al cierre del mercado del día d", "Último dato con fecha < t"],
    ["DXY", "Al cierre del mercado del día d", "Último dato con fecha < t"],
    ["TPM", "Se anuncia antes de entrar en vigencia", "Último dato con fecha < t"],
    ["DFF", "La Fed de Nueva York la publica la mañana de d+1", "Último dato con fecha < t − 1 (un día extra)"],
], columns=["variable", "cuándo se conoce el dato del día d", "regla aplicada"]).set_index("variable")
guardar_tabla(disponibilidad, "tab02_disponibilidad")
display(disponibilidad)

,cuándo se conoce el dato del día d,regla aplicada
variable,,
TRM_t,La tarde anterior a t (se calcula con las oper...,Se usa la de la propia fila t
Brent,Al cierre del mercado del día d,Último dato con fecha < t
DXY,Al cierre del mercado del día d,Último dato con fecha < t
TPM,Se anuncia antes de entrar en vigencia,Último dato con fecha < t
DFF,La Fed de Nueva York la publica la mañana de d+1,Último dato con fecha < t − 1 (un día extra)


## 2. Alineación: el último dato conocido, nunca el siguiente
Cada variable macro se une a la TRM con `pd.merge_asof(direction="backward", allow_exact_matches=False)`: se toma el **último** dato cuya fecha de disponibilidad es **estrictamente anterior** a *t*.
- **Si un día no hay dato** (festivo en Londres o Nueva York), se repite el último valor conocido. **Nunca se interpola**, porque interpolar usa el dato siguiente, que es futuro.
- **Esto precisa lo que la Entrega 1 llamó "forward-fill estrictamente hacia atrás":** el relleno siempre toma el valor conocido más reciente anterior a *t*.
- **Auditoría:** se guarda la fecha real del dato usado y su antigüedad en días, y una prueba automática detiene el notebook si alguna fila usa un dato no disponible.

In [3]:
alineada = alinear(trm, macros)
verificar_sin_fuga(alineada, macros)
print("Prueba anti-fuga superada: en todas las filas, cada dato macro estaba disponible antes de t.")
cols = ["fecha", "trm"] + [c for n in macros for c in (n, f"fecha_{n}", f"antig_{n}")]
display(alineada[cols].head(3))

Prueba anti-fuga superada: en todas las filas, cada dato macro estaba disponible antes de t.


,fecha,trm,brent,fecha_brent,antig_brent,dxy,fecha_dxy,antig_dxy,tpm,fecha_tpm,antig_tpm,dff,fecha_dff,antig_dff
0,2003-12-02,"2,829.0800",28.1700,2003-12-01,1,90.3500,2003-12-01,1,7.2500,2003-12-01,1,NaN,NaT,NaN
1,2003-12-03,"2,817.1400",29.1000,2003-12-02,1,89.6400,2003-12-02,1,7.2500,2003-12-02,1,1.0300,2003-12-01,2.0000
2,2003-12-04,"2,815.1900",29.2600,2003-12-03,1,89.5000,2003-12-03,1,7.2500,2003-12-03,1,0.9700,2003-12-02,2.0000


In [4]:
relleno = tabla_relleno(alineada, macros)
guardar_tabla(relleno, "tab02b_relleno")
display(relleno)

,rezago_publicacion_dias,filas_con_dato_repetido,pct_repetido,antiguedad_mediana_dias,antiguedad_p95_dias,antiguedad_max_dias,filas_con_dato_de_mas_de_5_dias
variable,,,,,,,
brent,0,129,2.3836,1.0000,1.0000,9.0000,14
dxy,0,10,0.1848,1.0000,1.0000,5.0000,0
tpm,0,0,0.0000,1.0000,1.0000,1.0000,0
dff,1,0,0.0000,2.0000,2.0000,2.0000,0


In [5]:
# Ejemplo concreto para la sustentación: una semana con festivo
ej = alineada[(alineada["fecha"] >= "2015-02-11") & (alineada["fecha"] <= "2015-02-19")]
display(ej[["fecha", "trm", "fecha_brent", "fecha_dxy", "fecha_tpm", "fecha_dff"]].assign(
    dia=ej["fecha"].dt.day_name()))

,fecha,trm,fecha_brent,fecha_dxy,fecha_tpm,fecha_dff,dia
2657,2015-02-11,"2,380.7900",2015-02-10,2015-02-10,2015-02-10,2015-02-09,Wednesday
2658,2015-02-12,"2,416.6100",2015-02-11,2015-02-11,2015-02-11,2015-02-10,Thursday
2659,2015-02-13,"2,401.0300",2015-02-12,2015-02-12,2015-02-12,2015-02-11,Friday
2660,2015-02-14,"2,376.2300",2015-02-13,2015-02-13,2015-02-13,2015-02-12,Saturday
2661,2015-02-18,"2,416.3700",2015-02-17,2015-02-17,2015-02-17,2015-02-16,Wednesday
2662,2015-02-19,"2,429.7100",2015-02-18,2015-02-18,2015-02-18,2015-02-17,Thursday


**Lectura:**
- **Casi siempre se usa el dato del día anterior.** La antigüedad mediana es de 1 día para Brent, DXY y TPM, y de 2 para la DFF por su día extra de publicación.
- **El relleno es poco y está explicado.** El Brent repite el último valor en 129 filas (2,4 %), porque Londres y Nueva York cierran en festivos distintos a los colombianos. Solo 14 filas usan un Brent de más de 5 días: Navidad, Año Nuevo, Semana Santa y el vacío de marzo de 2005 en FRED. El DXY repite en 10 filas y la TPM y la DFF, nunca.
- **Ejemplo de febrero de 2015, útil para la sustentación.** La TRM calculada el viernes 13 empieza a regir el sábado 14 y dura hasta el martes 17, porque el lunes 16 fue festivo en EE. UU. (Presidents' Day) y no hubo mercado cambiario. En esa fila se usan Brent, DXY y TPM del viernes 13, y la DFF del jueves 12. La fila siguiente es el miércoles 18 y usa datos del martes 17, y la DFF del lunes 16.

## 3. El objetivo y las variables de la Entrega 1
**Objetivo:** $y_t = \ln(\mathrm{TRM}_{t+1} / \mathrm{TRM}_t)$, el retorno de la siguiente TRM. Es el **único** cálculo del proyecto que mira hacia adelante (`shift(-1)`), porque es lo que se predice. El nivel en pesos se reconstruye como $\widehat{\mathrm{TRM}}_{t+1} = \mathrm{TRM}_t \cdot e^{\hat y}$.

**Variables:** las 6 de la Entrega 1, expresadas en retornos y calculadas solo hacia atrás (**Tabla 3**).

In [6]:
transformaciones = pd.DataFrame([
    ["y (objetivo)", "ln(TRM_{t+1} / TRM_t)", "Retorno de la siguiente TRM. Los árboles no extrapolan fuera del rango de entrenamiento; el retorno se mueve en un rango parecido en todas las épocas"],
    ["r_trm_l1", "ln(TRM_t / TRM_{t−1})", "Rezago de 1 día (Entrega 1). La TRM es un promedio del día: un movimiento suele continuar"],
    ["r_trm_l5", "ln(TRM_{t−4} / TRM_{t−5})", "Rezago de 5 días (Entrega 1): memoria de una semana hábil"],
    ["r_trm_l10", "ln(TRM_{t−9} / TRM_{t−10})", "Rezago de 10 días (Entrega 1): memoria de dos semanas hábiles"],
    ["r_brent", "ln(Brent_a(t) / Brent_a(t−1))", "Petróleo (Entrega 1), solo Brent: principal fuente de dólares; se espera signo negativo"],
    ["r_dxy", "ln(DXY_a(t) / DXY_a(t−1))", "Fortaleza global del dólar (Entrega 1); se espera signo positivo"],
    ["dif_tasas", "TPM_a(t) − DFF_a(t), en puntos porcentuales", "Diferencial Banrep–Fed (Entrega 1), en nivel porque es una tasa acotada; se espera signo negativo"],
], columns=["variable", "fórmula (a = valor alineado del paso 2)", "justificación"]).set_index("variable")
guardar_tabla(transformaciones, "tab03_transformaciones")
display(transformaciones)

,fórmula (a = valor alineado del paso 2),justificación
variable,,
y (objetivo),ln(TRM_{t+1} / TRM_t),Retorno de la siguiente TRM. Los árboles no ex...
r_trm_l1,ln(TRM_t / TRM_{t−1}),Rezago de 1 día (Entrega 1). La TRM es un prom...
r_trm_l5,ln(TRM_{t−4} / TRM_{t−5}),Rezago de 5 días (Entrega 1): memoria de una s...
r_trm_l10,ln(TRM_{t−9} / TRM_{t−10}),Rezago de 10 días (Entrega 1): memoria de dos ...
r_brent,ln(Brent_a(t) / Brent_a(t−1)),"Petróleo (Entrega 1), solo Brent: principal fu..."
r_dxy,ln(DXY_a(t) / DXY_a(t−1)),Fortaleza global del dólar (Entrega 1); se esp...
dif_tasas,"TPM_a(t) − DFF_a(t), en puntos porcentuales","Diferencial Banrep–Fed (Entrega 1), en nivel p..."


In [7]:
tabla = agregar_variables(alineada)
dataset = construir_dataset(trm, macros)
perdidas = len(tabla) - len(dataset)
print(f"Filas: {len(tabla):,} alineadas → {len(dataset):,} completas "
      f"({perdidas} perdidas: las primeras por los rezagos de hasta 10 TRM y la última por no tener TRM siguiente)")
print(f"Periodo del conjunto: {dataset['fecha'].min().date()} a {dataset['fecha'].max().date()}")
assert dataset[cfg.VARS_TODAS + ["y"]].notna().all().all()
display(dataset[cfg.VARS_TODAS + ["y"]].describe().T[["mean", "std", "min", "max"]])

Filas: 5,412 alineadas → 5,401 completas (11 perdidas: las primeras por los rezagos de hasta 10 TRM y la última por no tener TRM siguiente)
Periodo del conjunto: 2003-12-17 a 2026-09-26


,mean,std,min,max
r_trm_l1,0.0000,0.0078,-0.0562,0.0593
r_trm_l5,0.0000,0.0078,-0.0562,0.0593
r_trm_l10,0.0000,0.0078,-0.0562,0.0593
r_brent,0.0002,0.0276,-0.6437,0.4120
r_dxy,0.0000,0.0049,-0.0272,0.0252
dif_tasas,4.4745,2.0153,1.0000,9.8900
y,0.0000,0.0078,-0.0562,0.0593


## 4. Prueba de no fuga: alterar el futuro
Se alteran al azar **todos** los datos de las cinco series posteriores a una fecha y se reconstruye el conjunto desde cero. Si alguna variable mirara el futuro, cambiaría en filas anteriores a esa fecha. Como control, el objetivo de la última fila antes del corte **sí** debe cambiar, porque usa la TRM siguiente.

In [8]:
pruebas = pd.DataFrame([prueba_perturbacion(trm, macros, f, semilla=cfg.SEMILLA)
                        for f in ["2008-10-01", "2014-11-01", "2020-03-01", "2022-03-01"]]).set_index("fecha_corte")
assert (pruebas["valores_de_variables_que_cambiaron"] == 0).all()
assert (pruebas["objetivo_ultima_fila_original"] != pruebas["objetivo_ultima_fila_alterado"]).all()
guardar_tabla(pruebas, "tab03b_prueba_perturbacion")
display(pruebas)
print("En las 4 fechas de corte, ninguna variable cambió y el objetivo sí: no hay fuga.")

,filas_antes_del_corte,valores_de_variables_que_cambiaron,objetivo_ultima_fila_original,objetivo_ultima_fila_alterado
fecha_corte,,,,
2008-10-01,1140,0,-0.0086,0.2335
2014-11-01,2586,0,0.0073,0.2494
2020-03-01,3845,0,-0.0079,0.2343
2022-03-01,4320,0,-0.0022,0.2399


En las 4 fechas de corte, ninguna variable cambió y el objetivo sí: no hay fuga.


## 5. Partición cronológica: desarrollo y test
Como en el artículo y la Entrega 1: el **80 % inicial es desarrollo** y el **20 % final es test**, sin mezclar el orden. La última fila del desarrollo se **purga**, porque su objetivo usa la primera TRM del test. El test queda cerrado hasta el notebook 06.

Se guardan también el nivel de la TRM de hoy y el de mañana (`trm`, `trm_sig`): sirven para reconstruir el error en pesos y para reproducir el artículo, que predice niveles.

In [9]:
dataset["muestra"] = particion_cronologica(dataset)
conteo = dataset.groupby("muestra", sort=False)["fecha"].agg(filas="size", desde="min", hasta="max")
display(conteo)

dev, test = dataset[dataset["muestra"] == "desarrollo"], dataset[dataset["muestra"] == "test"]
assert dev["fecha"].max() < test["fecha"].min()
particion = {
    "fraccion_test": cfg.FRACCION_TEST,
    "filas_desarrollo": int(len(dev)), "filas_purga": int((dataset["muestra"] == "purga").sum()),
    "filas_test": int(len(test)),
    "desarrollo_desde": str(dev["fecha"].min().date()), "desarrollo_hasta": str(dev["fecha"].max().date()),
    "test_desde": str(test["fecha"].min().date()), "test_hasta": str(test["fecha"].max().date()),
    "trm_max_desarrollo": float(dev["trm"].max()),   # del test no se registra ningún valor: queda cerrado
}
columnas = ["fecha", "muestra", "trm", "trm_sig", "y"] + cfg.VARS_TODAS
dataset[columnas].to_csv(cfg.DATA_PROC / "dataset.csv", index=False)
(cfg.DATA_PROC / "particion.json").write_text(json.dumps(particion, indent=2), encoding="utf-8")
print(json.dumps(particion, indent=2))

,filas,desde,hasta
muestra,,,
desarrollo,4320,2003-12-17,2022-03-01
purga,1,2022-03-02,2022-03-02
test,1080,2022-03-03,2026-09-26


{
  "fraccion_test": 0.2,
  "filas_desarrollo": 4320,
  "filas_purga": 1,
  "filas_test": 1080,
  "desarrollo_desde": "2003-12-17",
  "desarrollo_hasta": "2022-03-01",
  "test_desde": "2022-03-03",
  "test_hasta": "2026-09-26",
  "trm_max_desarrollo": 4153.91
}


## Qué llevar al informe
- **Tabla 2** (`tab02_disponibilidad.csv`): cuándo se conoce cada dato y qué regla se aplica. Es el núcleo de la justificación anti-fuga (7.2). Se acompaña de la tabla de relleno (`tab02b_relleno.csv`).
- **Tabla 3** (`tab03_transformaciones.csv`): el objetivo y las 6 variables de la Entrega 1, con fórmula y justificación (7.1, transformaciones).
- **Prueba de perturbación** (`tab03b_prueba_perturbacion.csv`): en 4 fechas de corte, alterar todo el futuro no cambia ninguna variable y sí el objetivo.
- **Conjunto final:** 5.401 filas (2003-12-17 a 2026-09-26). **Desarrollo:** 4.320 filas hasta el 2022-03-01. **Test:** 1.080 filas desde el 2022-03-03. Se purga una fila entre ambos.
- **Dato clave para el notebook 04:** dentro del propio desarrollo, el nivel de la TRM cambia de rango con el tiempo: su máximo es **4.153,91** (2021), y antes de mediados de 2018 nunca había pasado de 3.435. Un modelo de árboles que predice niveles no puede superar lo que vio al entrenar, y ahí está el techo de extrapolación que justifica modelar retornos. Del test no se registra ningún valor en este notebook.
- **Para el informe:** la alineación precisa lo que la Entrega 1 llamó "forward-fill estrictamente hacia atrás". Siempre se usa el último dato conocido **antes** de *t*, y la DFF con un día más porque se publica al día siguiente.